# Portable execution configuration

This public copy was prepared from the executed JPCO revision notebook.

Machine-specific absolute paths and saved execution outputs were removed.
The original numerical summaries are provided separately in this revision
package.

By default, the notebook treats its current working directory as the
revision root. If required, set the following environment variables before
starting Jupyter:

- `GW_GLITCH_REVISION_ROOT`: directory containing the revision scripts and
  result subdirectories;
- `GW_GLITCH_PROJECT_ROOT`: parent `GW_Glitch` project directory containing
  historical processed datasets;
- `GW_GLITCH_RESEARCH_ROOT`: parent research directory;
- `GW_GLITCH_IMAGE_PROJECT_ROOT`: archived Gravity Spy image-project root.

The full source datasets, trained checkpoints, prediction arrays, and local
execution logs are not stored in Git.


In [ ]:
import os
from pathlib import Path

PORTABLE_REVISION_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_REVISION_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

PORTABLE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_PROJECT_ROOT",
        PORTABLE_REVISION_ROOT.parent.parent,
    )
).expanduser().resolve()

PORTABLE_RESEARCH_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_RESEARCH_ROOT",
        PORTABLE_PROJECT_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_DESKTOP_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_DESKTOP_ROOT",
        PORTABLE_RESEARCH_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_IMAGE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_IMAGE_PROJECT_ROOT",
        PORTABLE_RESEARCH_ROOT / "Glitch_Image",
    )
).expanduser().resolve()

print("Portable revision root:", PORTABLE_REVISION_ROOT)
print("Portable project root:", PORTABLE_PROJECT_ROOT)
print("Portable research root:", PORTABLE_RESEARCH_ROOT)
print("Portable image-project root:", PORTABLE_IMAGE_PROJECT_ROOT)


# Targeted Feature-Ablation Study

Controlled diagnostic experiment for the JPCO revision.

Models: XGBoost and custom MLP  
Split: frozen grouped-random split  
Seeds: 42, 43, and 44  
Hyperparameters: inherited from the completed revision experiments  
Test set: frozen and evaluated only after all four conditions are fixed  

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import sys

import numpy as np
import pandas as pd
import sklearn
import torch
import xgboost

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
).resolve()

DATA = ROOT / "jpco_o3_splits_v1"
OUT = ROOT / "targeted_ablation_v1"
OUT.mkdir(exist_ok=True)

EXPECTED_HASHES = {
    "manifest.json":
        "f1eedfaa282d6ae90e1567d271f9baff"
        "06c62ba233aff5890ae5dad85cfe9605",

    "o3_primary.csv.gz":
        "9d7d7d217563ccb121eba466fc442d83"
        "d7f25495e6e5ba40ea9a0962114829c6",

    "split_summary.csv":
        "61de488a44179ca86205ffcb45e98baa"
        "9eb2d9a7b52a1ab13cf2845368f88d01",

    "tuning_random_split.csv.gz":
        "8afbc1e03efa547ae4d51dcd3982308e"
        "baa67f018f2fc225b59daaa55f153342",
}

ORIGINAL_NINE = [
    "peak_time",
    "peak_time_ns",
    "start_time_ns",
    "duration",
    "peak_frequency",
    "central_freq",
    "amplitude",
    "snr",
    "q_value",
]

TIMESTAMP_FREE_SIX = [
    "duration",
    "peak_frequency",
    "central_freq",
    "amplitude",
    "snr",
    "q_value",
]

ABLATION_FEATURES = {
    # Remove only the absolute start timestamp.
    "remove_start_time_ns": [
        "peak_time",
        "peak_time_ns",
        "duration",
        "peak_frequency",
        "central_freq",
        "amplitude",
        "snr",
        "q_value",
    ],

    # Remove the two components representing peak time.
    "remove_peak_time_pair": [
        "start_time_ns",
        "duration",
        "peak_frequency",
        "central_freq",
        "amplitude",
        "snr",
        "q_value",
    ],

    # Begin from the timestamp-free six and remove amplitude and SNR.
    "timestamp_free_remove_amplitude_snr": [
        "duration",
        "peak_frequency",
        "central_freq",
        "q_value",
    ],

    # Begin from the timestamp-free six and remove Q.
    "timestamp_free_remove_q": [
        "duration",
        "peak_frequency",
        "central_freq",
        "amplitude",
        "snr",
    ],
}


def sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(block_size), b""):
            digest.update(block)

    return digest.hexdigest()


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Operating system:", platform.platform())
print("Architecture:", platform.machine())
print("Working directory:", Path.cwd())
print("Revision root:", ROOT)
print("Dataset directory:", DATA)
print("Output directory:", OUT)

print("\n--- Package versions ---")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("torch:", torch.__version__)

assert sys.version_info[:2] == (3, 12)
assert np.__version__ == "1.26.4"
assert pd.__version__ == "2.2.3"
assert sklearn.__version__ == "1.6.1"
assert xgboost.__version__ == "2.1.1"
assert torch.__version__.startswith("2.6.0")

print("\n--- Input hash verification ---")

verified_hashes = {}

for filename, expected in EXPECTED_HASHES.items():
    path = DATA / filename
    assert path.exists(), f"MISSING: {path}"

    observed = sha256(path)
    verified_hashes[filename] = observed

    print(filename)
    print("  observed:", observed)
    print("  expected:", expected)

    assert observed == expected, f"INPUT HASH MISMATCH: {filename}"

print("\nINPUT HASHES VERIFIED")

In [ ]:
manifest = json.loads((DATA / "manifest.json").read_text())
split_summary = pd.read_csv(DATA / "split_summary.csv")

print("--- Manifest feature sets ---")

for name, features in manifest["feature_sets"].items():
    print(f"{name}: {features}")

assert manifest["feature_sets"]["original_nine"] == ORIGINAL_NINE
assert manifest["feature_sets"]["timestamp_free_six"] == TIMESTAMP_FREE_SIX

print("\n--- Frozen grouped-random split counts ---")

random_counts = (
    split_summary.loc[
        split_summary["configuration"].eq("random_split")
    ]
    .sort_values("partition")
    .reset_index(drop=True)
)

display(random_counts)

count_map = dict(
    zip(
        random_counts["partition"],
        random_counts["rows"],
    )
)

assert int(count_map["train"]) == 280_163
assert int(count_map["validation"]) == 70_040
assert int(count_map["test"]) == 87_550

assert len(ORIGINAL_NINE) == 9
assert len(TIMESTAMP_FREE_SIX) == 6

expected_lengths = {
    "remove_start_time_ns": 8,
    "remove_peak_time_pair": 7,
    "timestamp_free_remove_amplitude_snr": 4,
    "timestamp_free_remove_q": 5,
}

for condition, features in ABLATION_FEATURES.items():
    assert len(features) == expected_lengths[condition]
    assert len(features) == len(set(features))
    assert set(features).issubset(ORIGINAL_NINE)

print("\n--- Frozen ablation definitions ---")

for condition, features in ABLATION_FEATURES.items():
    removed = [
        feature
        for feature in ORIGINAL_NINE
        if feature not in features
    ]

    reference = (
        "original_nine"
        if condition in {
            "remove_start_time_ns",
            "remove_peak_time_pair",
        }
        else "timestamp_free_six"
    )

    print(f"\n{condition}")
    print("  reference:", reference)
    print("  retained:", features)
    print("  removed:", removed)

print("\nFEATURE AND SPLIT CHECKS PASSED")

In [ ]:
print("--- Searching for selected model configurations ---")

selection_files = sorted(
    path
    for path in ROOT.rglob("selected_configuration.json")
    if "targeted_ablation_v1" not in str(path)
)

selection_records = []

for path in selection_files:
    path_text = str(path).lower()

    if "xgboost" not in path_text and "mlp" not in path_text:
        continue

    try:
        record = json.loads(path.read_text())
    except Exception as error:
        print(f"SKIPPED unreadable file: {path}")
        print(error)
        continue

    features = record.get("features", [])
    candidate = record.get("candidate")
    params = record.get("params")

    if features not in [ORIGINAL_NINE, TIMESTAMP_FREE_SIX]:
        continue

    model = (
        "XGBoost"
        if "xgboost" in path_text
        else "MLP"
    )

    feature_set = (
        "original_nine"
        if features == ORIGINAL_NINE
        else "timestamp_free_six"
    )

    item = {
        "model": model,
        "feature_set": feature_set,
        "candidate": candidate,
        "params": params,
        "path": str(path),
        "sha256": sha256(path),
    }

    selection_records.append(item)

    print("\nMODEL:", model)
    print("FEATURE SET:", feature_set)
    print("CANDIDATE:", candidate)
    print("PARAMETERS:", params)
    print("PATH:", path)
    print("SHA-256:", item["sha256"])

print(
    "\nSelection files located:",
    len(selection_records),
)

if len(selection_records) == 0:
    print(
        "\nNo selected_configuration.json files were found. "
        "This is not a dataset failure, but we must locate the "
        "stage-1 output directories before training."
    )
else:
    print("\nMODEL-SELECTION ARTIFACT SEARCH COMPLETE")

In [ ]:
print("--- Searching for completed baseline test results ---")

baseline_files = sorted(
    path
    for path in ROOT.rglob("test_per_seed.csv")
    if "targeted_ablation_v1" not in str(path)
)

baseline_records = []

required_conditions = {
    "random_split__original_nine",
    "random_split__timestamp_free_six",
}

for path in baseline_files:
    try:
        frame = pd.read_csv(path)
    except Exception:
        continue

    required_columns = {
        "condition",
        "seed",
        "test_weighted_f1",
    }

    if not required_columns.issubset(frame.columns):
        continue

    available = set(frame["condition"].astype(str))
    matching = available & required_conditions

    if not matching:
        continue

    path_text = str(path).lower()

    if "xgboost" in path_text:
        model = "XGBoost"
    elif "mlp" in path_text:
        model = "MLP"
    else:
        model = "unresolved"

    subset = frame.loc[
        frame["condition"].isin(required_conditions)
    ].copy()

    baseline_records.append(
        {
            "model": model,
            "path": str(path),
            "sha256": sha256(path),
            "rows": len(subset),
            "conditions": sorted(
                subset["condition"].unique().tolist()
            ),
        }
    )

    print("\nMODEL:", model)
    print("PATH:", path)
    print("SHA-256:", sha256(path))

    display(
        subset[
            [
                "condition",
                "seed",
                "test_weighted_f1",
            ]
        ].sort_values(["condition", "seed"])
    )

print("\nBaseline files located:", len(baseline_records))

if len(baseline_records) == 0:
    print(
        "\nNo baseline test_per_seed.csv files were found. "
        "We must locate the stage-2 output directories before "
        "calculating paired differences."
    )
else:
    print("\nBASELINE RESULT SEARCH COMPLETE")

In [ ]:
preflight_report = {
    "stage": "targeted feature-ablation preflight",
    "training_performed": False,
    "test_evaluated": False,
    "root": str(ROOT),
    "data_directory": str(DATA),
    "output_directory": str(OUT),
    "versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "torch": torch.__version__,
    },
    "verified_input_sha256": verified_hashes,
    "reference_features": {
        "original_nine": ORIGINAL_NINE,
        "timestamp_free_six": TIMESTAMP_FREE_SIX,
    },
    "ablation_features": ABLATION_FEATURES,
    "expected_runs": {
        "models": ["XGBoost", "MLP"],
        "conditions": list(ABLATION_FEATURES),
        "seeds": [42, 43, 44],
        "total_fits": 24,
    },
    "selection_artifacts_found": selection_records,
    "baseline_artifacts_found": baseline_records,
}

report_path = OUT / "preflight_report.json"

report_path.write_text(
    json.dumps(
        preflight_report,
        indent=2,
        sort_keys=True,
    )
)

print("REPORT SAVED:", report_path)
print("SHA-256:", sha256(report_path))
print("\nPREFLIGHT COMPLETE — NO TRAINING PERFORMED")

## Step 2 — XGBoost seed-42 pilot

Condition: remove only `start_time_ns`  
Reference configuration: grouped-random original-nine XGBoost  
Candidate: 7  
Seed: 42  
Evaluation at this stage: validation only  

In [ ]:
from pathlib import Path
import hashlib

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
).resolve()

SCRIPT = ROOT / "targeted_ablation_xgb_pilot.py"

EXPECTED_SCRIPT_SHA256 = (
    "db1ed9745207d0df4753a0c80094fd"
    "4269b6f9636bc827a742995a97108e1cb0"
)


def file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


assert SCRIPT.exists(), f"MISSING: {SCRIPT}"

observed = file_sha256(SCRIPT)

print("Script:", SCRIPT)
print("Observed SHA-256:", observed)
print("Expected SHA-256:", EXPECTED_SCRIPT_SHA256)

assert observed == EXPECTED_SCRIPT_SHA256

print("SCRIPT VERIFICATION PASSED")

In [ ]:
import subprocess
import sys
from datetime import datetime

LOG_DIR = ROOT / "run_logs"
LOG_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

log_path = (
    LOG_DIR /
    f"targeted_ablation_xgb_pilot_{timestamp}.log"
)

command = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--root",
    str(ROOT),
]

print("Command:")
print(" ".join(command))
print("Log:", log_path)
print()

with log_path.open("w") as log_stream:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log_stream.write(line)
        log_stream.flush()

    return_code = process.wait()

assert return_code == 0, (
    f"Pilot failed with exit code {return_code}. "
    f"Share the final log lines from {log_path}"
)

print("\nPILOT PROCESS EXITED SUCCESSFULLY")

In [ ]:
import json

PILOT_OUT = (
    ROOT /
    "targeted_ablation_v1" /
    "pilot_xgboost_remove_start_time_ns_seed42"
)

required = [
    PILOT_OUT / "protocol.json",
    PILOT_OUT / "pilot_summary.json",
    PILOT_OUT / "model.ubj",
    PILOT_OUT / "label_mapping.json",
    PILOT_OUT / "validation_predictions.npz",
]

for path in required:
    assert path.exists(), f"MISSING: {path}"
    assert path.stat().st_size > 0, f"EMPTY: {path}"

    print(
        f"READY: {path.name} "
        f"({path.stat().st_size:,} bytes)"
    )

protocol = json.loads(
    (PILOT_OUT / "protocol.json").read_text()
)

summary = json.loads(
    (PILOT_OUT / "pilot_summary.json").read_text()
)

assert protocol["condition"] == "remove_start_time_ns"
assert protocol["reference_feature_set"] == "original_nine"
assert protocol["split"] == "random_split"
assert protocol["seed"] == 42
assert protocol["candidate"] == 7
assert protocol["rounds"] == 200
assert len(protocol["features"]) == 8
assert "start_time_ns" not in protocol["features"]
assert protocol["test_evaluated"] is False

assert summary["completed"] is True
assert summary["model"] == "XGBoost"
assert summary["condition"] == "remove_start_time_ns"
assert summary["seed"] == 42
assert summary["train_rows"] == 280_163
assert summary["validation_rows"] == 70_040
assert summary["feature_count"] == 8
assert summary["test_evaluated"] is False

for name, expected_hash in summary[
    "artifact_sha256"
].items():
    observed_hash = file_sha256(PILOT_OUT / name)

    print(
        f"{name}: "
        f"{observed_hash == expected_hash}"
    )

    assert observed_hash == expected_hash

print("\nPILOT SUMMARY")
print(json.dumps(summary, indent=2))

print("\nCOMPLETION CHECK PASSED")

## Step 3 — Locate and verify authoritative training implementations

This step locates the exact XGBoost and MLP revision scripts used to generate
the selected configurations. No model training or test-set evaluation is performed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import platform
import sys

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
).resolve()

OUTPUT_DIR = ROOT / "targeted_ablation_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_SCRIPT_HASHES = {
    "XGBoost": (
        "7c06aa5cffb280dea07ec88ce4c140e9d6a98db6574996b60a377d0946c78b7b"
    ),
    "MLP": (
        "1aed088a3fa04eaed98c5fa081328b606a3085b43c2b3bd71a44bf1f14d18b93"
    ),
}

EXCLUDED_DIRECTORIES = {
    ".git",
    "__pycache__",
    "pilot_outputs",
    "workers",
}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def inspect_python_file(path):
    source = path.read_text(encoding="utf-8", errors="replace")
    tree = ast.parse(source, filename=str(path))

    classes = []
    functions = []

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            classes.append(node.name)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            functions.append(node.name)

    return {
        "classes": classes,
        "functions": functions,
    }


print("Revision root:", ROOT)
print("Searching Python files and computing SHA-256 hashes...")

python_files = []

for path in sorted(ROOT.rglob("*.py")):
    relative_parts = path.relative_to(ROOT).parts

    if any(part in EXCLUDED_DIRECTORIES for part in relative_parts):
        continue

    try:
        observed_hash = sha256_file(path)
    except (OSError, PermissionError):
        continue

    python_files.append(
        {
            "path": str(path),
            "relative_path": str(path.relative_to(ROOT)),
            "sha256": observed_hash,
            "bytes": path.stat().st_size,
        }
    )

print(f"Python files checked: {len(python_files):,}")

matches = {}

for model, expected_hash in EXPECTED_SCRIPT_HASHES.items():
    model_matches = [
        record
        for record in python_files
        if record["sha256"] == expected_hash
    ]

    matches[model] = model_matches

    print(f"\n{model}")
    print("Expected SHA-256:", expected_hash)

    if not model_matches:
        print("STATUS: NOT FOUND")
    else:
        for record in model_matches:
            path = Path(record["path"])
            structure = inspect_python_file(path)

            record["classes"] = structure["classes"]
            record["functions"] = structure["functions"]

            print("STATUS: FOUND")
            print("Path:", path)
            print("Bytes:", record["bytes"])
            print("Classes:", structure["classes"])
            print("Functions:", structure["functions"])


report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "completed": True,
    "training_performed": False,
    "test_evaluated": False,
    "root": str(ROOT),
    "python": sys.version,
    "platform": platform.platform(),
    "files_checked": len(python_files),
    "expected_script_hashes": EXPECTED_SCRIPT_HASHES,
    "matches": matches,
}

report_path = OUTPUT_DIR / "authoritative_script_search.json"

with report_path.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

report_hash = sha256_file(report_path)

print("\nREPORT SAVED:", report_path)
print("REPORT SHA-256:", report_hash)

missing = [
    model
    for model, model_matches in matches.items()
    if len(model_matches) == 0
]

duplicates = [
    model
    for model, model_matches in matches.items()
    if len(model_matches) > 1
]

if missing:
    print("\nSEARCH INCOMPLETE")
    print("Missing authoritative scripts:", missing)
    print("Do not launch the full ablation suite.")
elif duplicates:
    print("\nMULTIPLE IDENTICAL COPIES FOUND")
    print("Models with duplicate files:", duplicates)
    print("The paths must be resolved before training.")
else:
    print("\nAUTHORITATIVE IMPLEMENTATIONS VERIFIED")
    print("Exactly one hash-matching script was found for each model.")
    print("No training or test evaluation was performed.")

## Step 4 — Frozen full targeted-ablation suite

This stage evaluates XGBoost and MLP under four targeted feature ablations
using seeds 42–44. Hyperparameters are inherited from the appropriate
reference feature set without ablation-specific retuning. The frozen test
partition is evaluated once for every predetermined run.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
).resolve()

SCRIPT = ROOT / "targeted_ablation_full.py"

EXPECTED_HASHES = {
    "targeted_ablation_full.py":
        "41995316407f200dff733345b0ec08f0ac128c870a4386ed50a83d15df1ab7c5",
    "xgboost_revision_stage1.py":
        "7c06aa5cffb280dea07ec88ce4c140e9d6a98db6574996b60a377d0946c78b7b",
    "mlp_revision_stage1.py":
        "1aed088a3fa04eaed98c5fa081328b606a3085b43c2b3bd71a44bf1f14d18b93",
}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


print("Python:", sys.executable)
print("Revision root:", ROOT)
print()

for filename, expected_hash in EXPECTED_HASHES.items():
    path = ROOT / filename

    assert path.is_file(), f"Missing required file: {path}"

    observed_hash = sha256_file(path)

    print(filename)
    print("  observed:", observed_hash)
    print("  expected:", expected_hash)

    assert observed_hash == expected_hash, (
        f"Hash mismatch for {filename}. "
        "Do not launch the suite."
    )

search_report = (
    ROOT
    / "targeted_ablation_v1"
    / "authoritative_script_search.json"
)

assert search_report.is_file(), (
    f"Missing authoritative-script report: {search_report}"
)

report = json.loads(search_report.read_text())

assert report["completed"] is True
assert report["training_performed"] is False
assert report["test_evaluated"] is False
assert len(report["matches"]["XGBoost"]) == 1
assert len(report["matches"]["MLP"]) == 1

print()
print("PRE-LAUNCH VERIFICATION PASSED")
print("The runner and both authoritative implementations are unchanged.")

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import subprocess
import sys
import time

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
).resolve()

SCRIPT = ROOT / "targeted_ablation_full.py"

LOG_DIR = ROOT / "run_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

LOG_PATH = (
    LOG_DIR
    / f"targeted_ablation_full_{timestamp}.log"
)

command = [
    "/usr/bin/caffeinate",
    "-dimsu",
    sys.executable,
    "-u",
    str(SCRIPT),
    "--root",
    str(ROOT),
]

print("Command:")
print(" ".join(command))
print()
print("Log:")
print(LOG_PATH)
print()
print("Launching frozen 24-evaluation suite...")
print("The verified pilot will be reused; 23 models will be newly trained.")
print()

environment = os.environ.copy()

with LOG_PATH.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=environment,
        text=True,
    )

last_position = 0

while process.poll() is None:
    time.sleep(30)

    if LOG_PATH.exists():
        with LOG_PATH.open("r", encoding="utf-8") as handle:
            handle.seek(last_position)
            new_text = handle.read()
            last_position = handle.tell()

        if new_text:
            print(new_text, end="", flush=True)

return_code = process.wait()

# Print anything written after the final poll.
with LOG_PATH.open("r", encoding="utf-8") as handle:
    handle.seek(last_position)
    final_text = handle.read()

if final_text:
    print(final_text, end="", flush=True)

print()
print("Process exit code:", return_code)
print("Complete log:", LOG_PATH)

if return_code != 0:
    raise RuntimeError(
        "The targeted-ablation suite stopped before completion. "
        "Do not delete its output because the run is restart-safe. "
        f"Share the final output and this log: {LOG_PATH}"
    )

print()
print("FULL SUITE PROCESS EXITED SUCCESSFULLY")